<a href="https://colab.research.google.com/github/anaberereta-hue/Trabajos-Colab/blob/main/ordenamiento_busqueda_Grupo_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Tarea: Ordenamiento y busqueda

##Integrantes: Uriel Changoluisa - Francisco Narváez - Luciana Aldás - Anahí Real
##Fecha 21 de mayo 2026

# 🔢 Algoritmos de Ordenamiento y Búsqueda
> **Materia:** Estructura de Datos | **Lenguaje:** Python 3 | **Entorno:** Google Colab

---

## 📋 Estructura de la Guía

| Sección | Algoritmo | Tipo | Complejidad promedio |
|---------|-----------|------|---------------------|
| **A1** | Bubble Sort | Ordenamiento | O(n²) |
| **A2** | Merge Sort | Ordenamiento | O(n log n) |
| **A3** | Quick Sort | Ordenamiento | O(n log n) |
| **B1** | Búsqueda Lineal | Búsqueda | O(n) |
| **B2** | Búsqueda Binaria | Búsqueda | O(log n) |
| **B3** | Búsqueda por Hash | Búsqueda | O(1) promedio |

---

## 📌 Instrucciones para el Estudiante

Este notebook requiere tu participación activa. En **cada sección encontrarás celdas marcadas con 🖊️** donde deberás:

1. **Escribir documentación** explicando con tus propias palabras qué hace el algoritmo.
2. **Analizar la salida** de cada celda de ejecución y registrar tus observaciones.
3. **Completar ejercicios** de extensión al final de cada sección.

> ⚠️ **No avances** a la siguiente sección sin completar las celdas de documentación.

---
**Ejecuta las celdas en orden con `Shift+Enter`.**

In [ ]:
# ============================================================
# SETUP — Utilidades compartidas por todos los algoritmos
# ============================================================
# Este bloque define herramientas de visualización y medición
# de tiempo que se reutilizan en todas las secciones.
# Ejecútalo PRIMERO antes de cualquier otro bloque.

import time
import random
import copy

def generar_lista(n: int, rango: tuple = (1, 100), semilla: int = 42) -> list: #
    """
    Genera una lista de n enteros aleatorios en el rango dado.
    La semilla fija garantiza reproducibilidad entre ejecuciones.
    """
    random.seed(semilla)
    return [random.randint(*rango) for _ in range(n)]


def medir_tiempo(func, *args) -> tuple:
    """
    Ejecuta 'func(*args)' y retorna (resultado, tiempo_ms).
    Permite comparar el rendimiento de distintos algoritmos.
    """
    inicio = time.perf_counter()
    resultado = func(*args)
    fin = time.perf_counter()
    return resultado, round((fin - inicio) * 1000, 4)


def imprimir_lista(lst: list, etiqueta: str = '', max_items: int = 20) -> None:
    """
    Imprime una lista con etiqueta. Si supera max_items, muestra
    solo los primeros y últimos 5 elementos con '...' en el medio.
    """
    if len(lst) <= max_items:
        contenido = str(lst)
    else:
        inicio = str(lst[:5])[:-1]   # Quitar ']' del inicio
        fin    = str(lst[-5:])[1:]   # Quitar '[' del fin
        contenido = f"{inicio}, ..., {fin}"
    prefijo = f"{etiqueta}: " if etiqueta else ''
    print(f"  {prefijo}{contenido}  (len={len(lst)})")


def verificar_orden(original: list, ordenada: list) -> bool:
    """
    Verifica que 'ordenada' sea una permutación ordenada de 'original'.
    Compara contra sorted() como referencia.
    """
    return ordenada == sorted(original)


# Lista de prueba compartida (se copia para cada algoritmo)
LISTA_BASE = generar_lista(15, rango=(1, 99))
LISTA_GRANDE = generar_lista(5000, rango=(1, 10000))

print("✅ Utilidades cargadas correctamente.")
print()
imprimir_lista(LISTA_BASE, "Lista de prueba (n=15)")
imprimir_lista(LISTA_GRANDE, "Lista grande    (n=5000)")

✅ Utilidades cargadas correctamente.

  Lista de prueba (n=15): [82, 15, 4, 95, 36, 32, 29, 18, 95, 14, 87, 95, 70, 12, 76]  (len=15)
  Lista grande    (n=5000): [1825, 410, 4507, 4013, 3658, ..., 9420, 77, 4278, 2868, 6671]  (len=5000)


---
# 🔴 A1 — Bubble Sort (Ordenamiento Burbuja)

## Concepto
Bubble Sort compara elementos **adyacentes** y los intercambia si están en el orden incorrecto.
En cada pasada completa, el elemento más grande "burbujea" hasta su posición final al extremo derecho.

```
Pasada 1:  [64, 34, 25, 12, 22]  →  compara pares adyacentes
            [34, 25, 12, 22, 64]  →  64 llega al final
Pasada 2:  [25, 12, 22, 34, 64]  →  34 llega a su lugar
...
Final:     [12, 22, 25, 34, 64]  ✓
```

| Caso | Complejidad | Cuándo ocurre |
|------|-------------|---------------|
| Mejor | O(n) | Lista ya ordenada (con optimización) |
| Promedio | O(n²) | Lista aleatoria |
| Peor | O(n²) | Lista en orden inverso |
| Espacio | O(1) | Ordenamiento in-place |

In [ ]:
# ============================================================
# A1 — Paso 1: Implementación de Bubble Sort
# ============================================================
# ALGORITMO:
#   Para i = 0 hasta n-1:
#     Para j = 0 hasta n-i-2:
#       Si lista[j] > lista[j+1]:  →  intercambiar
#
# OPTIMIZACIÓN: si en una pasada no hubo intercambios,
# la lista ya está ordenada → terminar temprano (mejor caso O(n)).
#
# PARÁMETRO 'registrar_pasos': si True, guarda el estado de la
# lista después de cada pasada completa (útil para visualizar).

def bubble_sort(lista: list, registrar_pasos: bool = False) -> tuple:
    """
    Ordena una lista usando Bubble Sort (in-place).

    Args:
        lista:           Lista a ordenar (se modifica en el lugar).
        registrar_pasos: Si True, retorna historial de estados.
    Returns:
        Tupla (comparaciones, intercambios, pasos_opcionales).
    """
    n = len(lista)
    comparaciones  = 0
    intercambios   = 0
    pasos          = [lista[:]] if registrar_pasos else []  # Estado inicial

    for i in range(n):
        hubo_intercambio = False  # Bandera para la optimización

        # En la pasada i, los últimos i elementos ya están en su lugar
        for j in range(n - i - 1):
            comparaciones += 1

            if lista[j] > lista[j + 1]:
                # Intercambio sin variable temporal (tupla Python)
                lista[j], lista[j + 1] = lista[j + 1], lista[j]
                intercambios += 1
                hubo_intercambio = True

        if registrar_pasos:
            pasos.append(lista[:])  # Capturar estado tras cada pasada

        # Optimización: lista ya ordenada → salir temprano
        if not hubo_intercambio:
            break

    return comparaciones, intercambios, pasos


print("✅ bubble_sort() definida.")

✅ bubble_sort() definida.


In [ ]:
# ============================================================
# A1 — Paso 2: Traza visual de Bubble Sort
# ============================================================
# Mostramos cómo cambia la lista después de cada pasada completa.
# Esto permite observar cómo los elementos grandes se desplazan
# progresivamente hacia la derecha.

datos = copy.copy(LISTA_BASE)  # Copia para no modificar la original

print("=" * 60)
print(" A1 — Bubble Sort: traza por pasadas")
print("=" * 60)
print(f"  Lista original: {datos}")
print()

comp, intercambios, pasos = bubble_sort(datos, registrar_pasos=True)

for i, estado in enumerate(pasos):
    etiqueta = "Inicio " if i == 0 else f"Pasada {i:>2}"
    # Marcar el elemento que llegó a su posición final en esta pasada
    print(f"  {etiqueta}: {estado}")

print()
print("-" * 60)
print(f"  Pasadas totales:   {len(pasos) - 1}")
print(f"  Comparaciones:     {comp}")
print(f"  Intercambios:      {intercambios}")
estado_ok = verificar_orden(LISTA_BASE, datos)
print(f"  Resultado correcto: {'✅ Sí' if estado_ok else '❌ No'}")

 A1 — Bubble Sort: traza por pasadas
  Lista original: [82, 15, 4, 95, 36, 32, 29, 18, 95, 14, 87, 95, 70, 12, 76]

  Inicio : [82, 15, 4, 95, 36, 32, 29, 18, 95, 14, 87, 95, 70, 12, 76]
  Pasada  1: [15, 4, 82, 36, 32, 29, 18, 95, 14, 87, 95, 70, 12, 76, 95]
  Pasada  2: [4, 15, 36, 32, 29, 18, 82, 14, 87, 95, 70, 12, 76, 95, 95]
  Pasada  3: [4, 15, 32, 29, 18, 36, 14, 82, 87, 70, 12, 76, 95, 95, 95]
  Pasada  4: [4, 15, 29, 18, 32, 14, 36, 82, 70, 12, 76, 87, 95, 95, 95]
  Pasada  5: [4, 15, 18, 29, 14, 32, 36, 70, 12, 76, 82, 87, 95, 95, 95]
  Pasada  6: [4, 15, 18, 14, 29, 32, 36, 12, 70, 76, 82, 87, 95, 95, 95]
  Pasada  7: [4, 15, 14, 18, 29, 32, 12, 36, 70, 76, 82, 87, 95, 95, 95]
  Pasada  8: [4, 14, 15, 18, 29, 12, 32, 36, 70, 76, 82, 87, 95, 95, 95]
  Pasada  9: [4, 14, 15, 18, 12, 29, 32, 36, 70, 76, 82, 87, 95, 95, 95]
  Pasada 10: [4, 14, 15, 12, 18, 29, 32, 36, 70, 76, 82, 87, 95, 95, 95]
  Pasada 11: [4, 14, 12, 15, 18, 29, 32, 36, 70, 76, 82, 87, 95, 95, 95]
  Pasada 1

In [ ]:
# ============================================================
# A1 — Paso 3: Benchmark — Mejor, Promedio y Peor caso
# ============================================================
# Medimos el tiempo de Bubble Sort en tres escenarios:
#   - Mejor caso:  lista ya ordenada (la optimización actúa)
#   - Promedio:    lista aleatoria
#   - Peor caso:   lista en orden inverso (máximos intercambios)

N = 2000  # Tamaño para el benchmark

lista_random  = generar_lista(N)
lista_sorted  = sorted(lista_random)
lista_reverse = list(reversed(lista_sorted))

_, t_mejor   = medir_tiempo(bubble_sort, lista_sorted[:])
_, t_promedio= medir_tiempo(bubble_sort, lista_random[:])
_, t_peor    = medir_tiempo(bubble_sort, lista_reverse[:])

print("=" * 55)
print(f" A1 — Bubble Sort Benchmark  (n={N})")
print("=" * 55)
print(f"  Mejor caso  (ordenada):  {t_mejor:>8.4f} ms")
print(f"  Promedio    (aleatoria): {t_promedio:>8.4f} ms")
print(f"  Peor caso   (inversa):   {t_peor:>8.4f} ms")
print()
print(f"  Factor peor/mejor: {t_peor/t_mejor:.1f}x")
print()
print("📌 Observa que el peor caso es ~n veces más lento que el mejor.")
print("   Esto confirma la complejidad O(n²) vs O(n) con la optimización.")

 A1 — Bubble Sort Benchmark  (n=2000)
  Mejor caso  (ordenada):    0.3436 ms
  Promedio    (aleatoria): 580.0647 ms
  Peor caso   (inversa):   786.4367 ms

  Factor peor/mejor: 2288.8x

📌 Observa que el peor caso es ~n veces más lento que el mejor.
   Esto confirma la complejidad O(n²) vs O(n) con la optimización.


## 🖊️ Documentación Estudiantil — A1 Bubble Sort

> **Instrucción:** Completa las siguientes secciones con tus propias palabras.
> Edita esta celda haciendo doble clic sobre ella.

---

### 1. Explicación del algoritmo
*¿Cómo funciona Bubble Sort? Describe el proceso con tus propias palabras:*

```
El algoritmo funciona atraves una revisión secuencial de los elementos de la lista en grupos de dos, desde el extremo izquierdo. Este consiste en comparar el primero con el segundo, y si el elemento de la izquierda es mayor que el segundo, se realiza un intercambio de posiciones. De lo contrario no se realiza el intercambio. Después se avanza una posición hacia la derecha para evaluar los siguientes dos elementes (el segundo y el tercero), repitiendo este sistema hasta llegar al final de la lista. Dado que este recorrido debe realizarse varias veces para verificar el orden, el codigo tiene en cuenta que si no se realizó ningun cambio entonces finaliza rapidamente ya que ya esta ordenado.
```

### 2. Análisis de la salida
*Observa la traza por pasadas. ¿Qué patrón notas en los elementos al final de cada pasada?*

```
El patrón constante que se nota es el desplazamiento del número mayor disponible hacia el extremo derecho en cada movimiento. En la primera pasada, el número con el valor más alto queda en las ultimas posiciones y así sucesivamente en las siguientes pasadas. Esto refleja que la lista se va ordenando de manera corrida e inversa, colocando los numeros en su posicion correspondiente, lo que reduce el número de comparaciones necesarias en las pasadas sigueintes al no requerir la revisión de las posiciones ya ordenadas.
```

### 3. ¿Para qué casos usarías Bubble Sort?
*Considerando sus complejidades, ¿cuándo tiene sentido usarlo y cuándo no?*

```
Es recomendable usarlo cuando al aplicarlo, la lista este mayormente ordenada, y solo se haría ciertos intercambios, además que sea una cantidad considerable y manejable de datos o numeros. En este último caso, el mecanismo de parada temprana permite que el proceso concluya con alta eficiencia.

Y no es recomendable aplicar este algoritmos en cantidad de datos de mayor volumen, ya que, se necesita evaluar dos numero por dos numeros hasta acabar la lista, por lo que tomaria mucho tiempo al emplearlo.
```

### 4. Resultado del benchmark
*Registra los tiempos medidos y explica la diferencia entre los tres casos:*

| Caso | Tiempo (ms) | Explicación |
|------|------------|-------------|
| Mejor | 0.3436| Es el más rápido porque la lista ya estaba ordenada. El algoritmo solo revisó las parejas una sola vez, activó su break y terminó el trabajo de inmediato. |
| Promedio | 580.1 | Es notablemente más lento. Al estar los números mezclados al azar, el algoritmo tuvo que dar muchas vueltas completas a la lista, comparando parejas y cambiando elementos de lugar constantemente hasta que todos se acomodaron. |
| Peor | 786.4 | Es el caso más lento de todos, como la lista estaba al revés, el algoritmo tuvo que realizar el máximo número posible de comparaciones y tuvo que mover absolutamente todos los números en cada paso para llevarlos desde el principio hasta el final. |

---
# 🟢 A2 — Merge Sort (Ordenamiento por Mezcla)

## Concepto
Merge Sort aplica la estrategia **Dividir y Conquistar**: divide la lista en mitades recursivamente
hasta tener sublistas de 1 elemento (ya ordenadas por definición), luego las **fusiona** ordenadamente.

```
[38, 27, 43, 3]           →  DIVIDIR
[38, 27]  |  [43, 3]      →  DIVIDIR
[38] [27] | [43] [3]      →  casos base
[27, 38]  |  [3, 43]      →  FUSIONAR
[3, 27, 38, 43]           →  FUSIONAR
```

| Caso | Complejidad | Espacio |
|------|-------------|---------|
| Todos | O(n log n) | O(n) auxiliar |

In [ ]:
# ============================================================
# A2 — Paso 1: Implementación de Merge Sort
# ============================================================
# Merge Sort tiene DOS funciones:
#
#   merge_sort(lista):  función principal recursiva que DIVIDE.
#   fusionar(izq, der): función auxiliar que FUSIONA dos sublistas
#                       ya ordenadas en una sola lista ordenada.
#
# La recursión sigue el árbol:
#   merge_sort([38,27,43,3])
#     merge_sort([38,27])        merge_sort([43,3])
#       merge_sort([38])           merge_sort([43])
#       merge_sort([27])           merge_sort([3])
#       fusionar([38],[27])        fusionar([43],[3])
#     fusionar([27,38],[3,43])

comparaciones_merge = 0   # Contador global para esta ejecución

def fusionar(izq: list, der: list) -> list:
    """
    Fusiona dos sublistas ordenadas en una sola lista ordenada.

    Estrategia: dos punteros, uno en cada sublista.
    Tomamos el menor de los dos elementos apuntados y avanzamos
    el puntero correspondiente. Al agotar una sublista, agregamos
    el resto de la otra directamente (ya están ordenados).

    Complejidad: O(n) donde n = len(izq) + len(der).
    """
    global comparaciones_merge
    resultado = []
    i = j = 0

    # Comparar elemento a elemento y tomar el menor
    while i < len(izq) and j < len(der):
        comparaciones_merge += 1
        if izq[i] <= der[j]:
            resultado.append(izq[i])
            i += 1
        else:
            resultado.append(der[j])
            j += 1

    # Agregar los elementos restantes (ya ordenados)
    resultado.extend(izq[i:])
    resultado.extend(der[j:])
    return resultado


def merge_sort(lista: list) -> list:
    """
    Ordena una lista usando Merge Sort (divide y conquista).

    Args:
        lista: Lista a ordenar.
    Returns:
        Nueva lista ordenada (no modifica la original).
    """
    # Caso base: lista de 0 o 1 elemento ya está ordenada
    if len(lista) <= 1:
        return lista

    # Dividir en dos mitades
    mid  = len(lista) // 2
    izq  = merge_sort(lista[:mid])   # Ordenar mitad izquierda
    der  = merge_sort(lista[mid:])   # Ordenar mitad derecha

    # Fusionar las dos mitades ya ordenadas
    return fusionar(izq, der)


# Demo rápida
datos = copy.copy(LISTA_BASE)
comparaciones_merge = 0
resultado = merge_sort(datos)

print("=" * 55)
print(" A2 — Merge Sort")
print("=" * 55)
print(f"  Original:  {datos}")
print(f"  Ordenada:  {resultado}")
print(f"  Correcta:  {'✅ Sí' if verificar_orden(datos, resultado) else '❌ No'}")
print(f"  Comparaciones (n={len(datos)}): {comparaciones_merge}")
print(f"  n·log2(n) aprox.:              {len(datos) * (len(datos)).bit_length():.0f}")

 A2 — Merge Sort
  Original:  [82, 15, 4, 95, 36, 32, 29, 18, 95, 14, 87, 95, 70, 12, 76]
  Ordenada:  [4, 12, 14, 15, 18, 29, 32, 36, 70, 76, 82, 87, 95, 95, 95]
  Correcta:  ✅ Sí
  Comparaciones (n=15): 43
  n·log2(n) aprox.:              60


In [ ]:
# ============================================================
# A2 — Paso 2: Visualización del árbol de recursión
# ============================================================
# Mostramos la secuencia de llamadas recursivas para una lista
# pequeña, con indentación que representa la profundidad del árbol.

def merge_sort_verbose(lista: list, nivel: int = 0) -> list:
    """Versión de Merge Sort que imprime el árbol de llamadas."""
    sangria = "  " * nivel
    print(f"{sangria}↓ merge_sort({lista})")

    if len(lista) <= 1:
        print(f"{sangria}  → caso base: {lista}")
        return lista

    mid = len(lista) // 2
    izq = merge_sort_verbose(lista[:mid], nivel + 1)
    der = merge_sort_verbose(lista[mid:], nivel + 1)

    fusionado = fusionar(izq, der)
    print(f"{sangria}↑ fusionar({izq}, {der}) → {fusionado}")
    return fusionado


print("=" * 55)
print(" A2 — Árbol de recursión de Merge Sort")
print("=" * 55)
comparaciones_merge = 0
_ = merge_sort_verbose([38, 27, 43, 3, 9, 82, 10])
print()
print("📌 Las flechas ↓ indican división (bajar en el árbol).")
print("   Las flechas ↑ indican fusión (subir en el árbol).")

 A2 — Árbol de recursión de Merge Sort
↓ merge_sort([38, 27, 43, 3, 9, 82, 10])
  ↓ merge_sort([38, 27, 43])
    ↓ merge_sort([38])
      → caso base: [38]
    ↓ merge_sort([27, 43])
      ↓ merge_sort([27])
        → caso base: [27]
      ↓ merge_sort([43])
        → caso base: [43]
    ↑ fusionar([27], [43]) → [27, 43]
  ↑ fusionar([38], [27, 43]) → [27, 38, 43]
  ↓ merge_sort([3, 9, 82, 10])
    ↓ merge_sort([3, 9])
      ↓ merge_sort([3])
        → caso base: [3]
      ↓ merge_sort([9])
        → caso base: [9]
    ↑ fusionar([3], [9]) → [3, 9]
    ↓ merge_sort([82, 10])
      ↓ merge_sort([82])
        → caso base: [82]
      ↓ merge_sort([10])
        → caso base: [10]
    ↑ fusionar([82], [10]) → [10, 82]
  ↑ fusionar([3, 9], [10, 82]) → [3, 9, 10, 82]
↑ fusionar([27, 38, 43], [3, 9, 10, 82]) → [3, 9, 10, 27, 38, 43, 82]

📌 Las flechas ↓ indican división (bajar en el árbol).
   Las flechas ↑ indican fusión (subir en el árbol).


In [ ]:
# ============================================================
# A2 — Paso 3: Comparación Bubble Sort vs Merge Sort
# ============================================================
# Ordenamos listas de distintos tamaños con ambos algoritmos
# y comparamos sus tiempos. Esto ilustra de forma práctica
# la diferencia entre O(n²) y O(n log n).

tamanios = [100, 500, 1000, 2000, 5000]

print("=" * 60)
print(" A2 — Bubble Sort vs Merge Sort (tiempos en ms)")
print("=" * 60)
print(f"  {'n':>6}  {'Bubble Sort':>14}  {'Merge Sort':>12}  {'Aceleración':>12}")
print("-" * 60)

for n in tamanios:
    datos = generar_lista(n)
    _, t_bubble = medir_tiempo(bubble_sort, datos[:])
    comparaciones_merge = 0
    _, t_merge  = medir_tiempo(merge_sort,  datos[:])
    factor = t_bubble / t_merge if t_merge > 0 else float('inf')
    print(f"  {n:>6}  {t_bubble:>14.4f}  {t_merge:>12.4f}  {factor:>10.1f}x")

print()
print("📌 Con n=5000, Merge Sort es significativamente más rápido.")
print("   El factor de aceleración crece con n, consistente con O(n²) vs O(n log n).")

 A2 — Bubble Sort vs Merge Sort (tiempos en ms)
       n     Bubble Sort    Merge Sort   Aceleración
------------------------------------------------------------
     100          0.5048        0.2307         2.2x
     500         21.7578        1.0290        21.1x
    1000         88.4731        2.3325        37.9x
    2000        284.9686        4.8612        58.6x
    5000       1825.6672       14.0493       129.9x

📌 Con n=5000, Merge Sort es significativamente más rápido.
   El factor de aceleración crece con n, consistente con O(n²) vs O(n log n).


## 🖊️ Documentación Estudiantil — A2 Merge Sort

> **Instrucción:** Edita esta celda haciendo doble clic.

---

### 1. Explica el principio Dividir y Conquistar
*¿En qué consiste esta estrategia? ¿Cómo la aplica Merge Sort?*

```
[ESCRIBE AQUÍ]
```

### 2. Explica la función fusionar()
*¿Qué hace exactamente? ¿Por qué es O(n)?*

```
[ESCRIBE AQUÍ]
```

### 3. Análisis del árbol de recursión
*¿Cuántos niveles tiene el árbol para una lista de 7 elementos? ¿Por qué?*

```
[ESCRIBE AQUÍ]
```

### 4. Tabla comparativa de tiempos
*Copia los resultados del benchmark y analiza la tendencia:*

| n | Bubble (ms) | Merge (ms) | Factor | Mi análisis |
|---|-------------|------------|--------|-----------|
| 100 | | | | |
| 1000 | | | | |
| 5000 | | | | |

---
# 🔵 A3 — Quick Sort (Ordenamiento Rápido)

## Concepto
Quick Sort también usa **Dividir y Conquistar**, pero con una estrategia diferente:
elige un elemento **pivote** y reorganiza la lista de modo que todos los elementos
menores al pivote queden a su izquierda y los mayores a su derecha.

```
Pivote = 3
[3, 6, 8, 10, 1, 2, 1]
[1, 2, 1]  [3]  [6, 8, 10]   →  particionar
↑ Quick Sort recursivo sobre cada parte
```

| Caso | Complejidad | Cuándo ocurre |
|------|-------------|---------------|
| Mejor / Promedio | O(n log n) | Pivote divide equitativamente |
| Peor | O(n²) | Pivote siempre min o max (lista ordenada) |
| Espacio | O(log n) | Pila de llamadas recursivas |

In [ ]:
# ============================================================
# A3 — Paso 1: Implementación de Quick Sort
# ============================================================
# ESTRATEGIA DE PIVOTE: elegimos el elemento MEDIANO entre el
# primer, último y elemento central. Esta heurística evita
# el peor caso O(n²) en listas ya ordenadas.
#
# PARTICIÓN:
#   Reorganizamos in-place usando el esquema de Lomuto:
#   - 'i' apunta al límite de elementos menores que el pivote
#   - Recorremos con 'j': si lista[j] < pivote, lo movemos a la
#     zona izquierda incrementando i y haciendo swap.

comparaciones_quick = 0  # Contador global

def elegir_pivote_mediana(lista: list, bajo: int, alto: int) -> int:
    """
    Elige el pivote como la mediana entre lista[bajo], lista[medio] y lista[alto].
    Coloca el pivote en lista[alto-1] y retorna su valor.
    Esta estrategia mejora el rendimiento en listas ordenadas o casi ordenadas.
    """
    medio = (bajo + alto) // 2
    # Ordenar los tres candidatos para encontrar la mediana
    if lista[bajo] > lista[medio]:
        lista[bajo], lista[medio] = lista[medio], lista[bajo]
    if lista[bajo] > lista[alto]:
        lista[bajo], lista[alto] = lista[alto], lista[bajo]
    if lista[medio] > lista[alto]:
        lista[medio], lista[alto] = lista[alto], lista[medio]
    # El pivote (mediana) queda en lista[medio]; lo movemos al final-1
    lista[medio], lista[alto - 1] = lista[alto - 1], lista[medio]
    return lista[alto - 1]


def particionar(lista: list, bajo: int, alto: int) -> int:
    """
    Particiona lista[bajo..alto] alrededor del pivote.
    Retorna el índice final del pivote.
    """
    global comparaciones_quick
    if alto - bajo < 2:   # Lista de 2 elementos: no hay pivote mediana
        pivote = lista[alto]
    else:
        pivote = elegir_pivote_mediana(lista, bajo, alto)

    i = bajo - 1  # Límite de elementos menores

    for j in range(bajo, alto):
        comparaciones_quick += 1
        if lista[j] <= pivote:
            i += 1
            lista[i], lista[j] = lista[j], lista[i]

    # Colocar el pivote en su posición definitiva
    lista[i + 1], lista[alto] = lista[alto], lista[i + 1]
    return i + 1  # Índice definitivo del pivote


def quick_sort(lista: list, bajo: int = 0, alto: int = None) -> None:
    """
    Ordena lista[bajo..alto] usando Quick Sort in-place.

    Args:
        lista: Lista a ordenar (se modifica en el lugar).
        bajo:  Índice inferior del rango a ordenar (default 0).
        alto:  Índice superior del rango a ordenar (default len-1).
    """
    if alto is None:
        alto = len(lista) - 1

    if bajo < alto:
        # Particionar: el pivote queda en su posición final
        idx_pivote = particionar(lista, bajo, alto)

        # Ordenar recursivamente la mitad izquierda (excluye el pivote)
        quick_sort(lista, bajo, idx_pivote - 1)
        # Ordenar recursivamente la mitad derecha (excluye el pivote)
        quick_sort(lista, idx_pivote + 1, alto)


# Demo rápida
datos = copy.copy(LISTA_BASE)
original = datos[:]
comparaciones_quick = 0
quick_sort(datos)

print("=" * 55)
print(" A3 — Quick Sort")
print("=" * 55)
print(f"  Original:  {original}")
print(f"  Ordenada:  {datos}")
print(f"  Correcta:  {'✅ Sí' if verificar_orden(original, datos) else '❌ No'}")
print(f"  Comparaciones (n={len(original)}): {comparaciones_quick}")

 A3 — Quick Sort
  Original:  [82, 15, 4, 95, 36, 32, 29, 18, 95, 14, 87, 95, 70, 12, 76]
  Ordenada:  [4, 12, 18, 14, 15, 29, 32, 76, 36, 70, 82, 87, 95, 95, 95]
  Correcta:  ❌ No
  Comparaciones (n=15): 47


In [ ]:
# ============================================================
# A3 — Paso 2: Visualización de particiones
# ============================================================
# Mostramos qué ocurre en cada llamada a particionar().
# Se imprime el estado de la lista y el pivote seleccionado
# en cada nivel de recursión.

llamadas_viz = []

def quick_sort_verbose(lista: list, bajo: int = 0, alto: int = None, nivel: int = 0) -> None:
    """Quick Sort con traza de cada partición."""
    if alto is None:
        alto = len(lista) - 1

    if bajo < alto:
        sangria = "  " * nivel
        sub = lista[bajo:alto+1]
        print(f"{sangria}Particionando: {sub}  (índices {bajo}..{alto})")

        idx = particionar(lista, bajo, alto)
        pivote = lista[idx]

        izq = lista[bajo:idx]
        der = lista[idx+1:alto+1]
        print(f"{sangria}  Pivote={pivote} → izq={izq} | [{pivote}] | der={der}")

        quick_sort_verbose(lista, bajo, idx - 1, nivel + 1)
        quick_sort_verbose(lista, idx + 1, alto, nivel + 1)


print("=" * 60)
print(" A3 — Quick Sort: traza de particiones")
print("=" * 60)
muestra = [10, 80, 30, 90, 40, 50, 70]
print(f"  Lista inicial: {muestra}")
print()
comparaciones_quick = 0
quick_sort_verbose(muestra)
print()
print(f"  Resultado final: {muestra}")
print()
print("📌 Cada pivote queda fijo en su posición definitiva.")
print("   Los subproblemas de la izquierda y derecha se resuelven de forma independiente.")

 A3 — Quick Sort: traza de particiones
  Lista inicial: [10, 80, 30, 90, 40, 50, 70]

Particionando: [10, 80, 30, 90, 40, 50, 70]  (índices 0..6)
  Pivote=90 → izq=[10, 30, 50, 40, 70] | [90] | der=[80]
  Particionando: [10, 30, 50, 40, 70]  (índices 0..4)
    Pivote=70 → izq=[10, 30, 40, 50] | [70] | der=[]
    Particionando: [10, 30, 40, 50]  (índices 0..3)
      Pivote=50 → izq=[10, 30] | [50] | der=[40]
      Particionando: [10, 30]  (índices 0..1)
        Pivote=30 → izq=[10] | [30] | der=[]

  Resultado final: [10, 30, 50, 40, 70, 90, 80]

📌 Cada pivote queda fijo en su posición definitiva.
   Los subproblemas de la izquierda y derecha se resuelven de forma independiente.


In [ ]:
# ============================================================
# A3 — Paso 3: Comparación de los tres algoritmos
# ============================================================

tamanios = [500, 1000, 2000, 5000]

print("=" * 75)
print(" Comparación final: Bubble Sort vs Merge Sort vs Quick Sort")
print("=" * 75)
print(f"  {'n':>5}  {'Bubble (ms)':>12}  {'Merge (ms)':>11}  {'Quick (ms)':>11}  {'Mejor':>8}")
print("-" * 75)

for n in tamanios:
    datos = generar_lista(n)
    _, tb = medir_tiempo(bubble_sort, datos[:])
    comparaciones_merge = 0
    _, tm = medir_tiempo(merge_sort,  datos[:])
    comparaciones_quick = 0
    _, tq = medir_tiempo(quick_sort,  datos[:])
    mejor = min([(tb,'Bubble'),(tm,'Merge'),(tq,'Quick')], key=lambda x:x[0])[1]
    print(f"  {n:>5}  {tb:>12.4f}  {tm:>11.4f}  {tq:>11.4f}  {mejor:>8}")

print()
print("📌 Quick Sort suele ganar en listas aleatorias por sus bajos costos de cache.")
print("   Merge Sort garantiza O(n log n) en todos los casos, Quick Sort no.")

 Comparación final: Bubble Sort vs Merge Sort vs Quick Sort
      n   Bubble (ms)   Merge (ms)   Quick (ms)     Mejor
---------------------------------------------------------------------------
    500       17.9592       1.0866       0.8957     Quick
   1000       65.3921       2.4500       2.4751     Merge
   2000      405.2343      10.5254      13.6812     Merge
   5000     3127.7196      13.8998      28.7968     Merge

📌 Quick Sort suele ganar en listas aleatorias por sus bajos costos de cache.
   Merge Sort garantiza O(n log n) en todos los casos, Quick Sort no.


## 🖊️ Documentación Estudiantil — A3 Quick Sort

> Edita esta celda haciendo doble clic.

---

### 1. ¿Qué es el pivote y para qué sirve?

```
[ESCRIBE AQUÍ]
```

### 2. ¿Por qué el peor caso de Quick Sort es O(n²)?
*¿En qué situación ocurre? ¿Cómo lo mitiga la estrategia de mediana?*

```
[ESCRIBE AQUÍ]
```

### 3. Diferencias entre Quick Sort y Merge Sort
*Completa la tabla:*

| Característica | Quick Sort | Merge Sort |
|----------------|-----------|------------|
| In-place | | |
| Estable | | |
| Peor caso | | |
| Uso de memoria | | |

### 4. Resultados del benchmark de tres algoritmos
*¿En qué tamaño de lista empieza a notarse la diferencia entre O(n²) y O(n log n)?*

```
[ESCRIBE AQUÍ]
```

---
# 🔍 SECCIÓN B — Algoritmos de Búsqueda

> Buscar significa encontrar la posición (o confirmar la ausencia) de un elemento
> en una colección de datos.

| Algoritmo | Requiere | Complejidad | Memoria |
|-----------|----------|-------------|--------|
| **B1** Lineal | Nada | O(n) | O(1) |
| **B2** Binaria | Lista **ordenada** | O(log n) | O(1) |
| **B3** Hash | Tabla hash | O(1) promedio | O(n) |

---
# 🔴 B1 — Búsqueda Lineal (Sequential Search)

## Concepto
La búsqueda lineal recorre la lista elemento a elemento desde el inicio
hasta encontrar el objetivo o agotar la lista. Es el método más simple
y funciona en **listas sin orden previo**.

```
Buscar 43 en [12, 5, 43, 7, 99, 2]
  12 ≠ 43  → siguiente
   5 ≠ 43  → siguiente
  43 = 43  → ¡Encontrado en índice 2!
```

In [ ]:
# ============================================================
# B1 — Paso 1: Búsqueda Lineal — implementación y variantes
# ============================================================
# Implementamos tres variantes de búsqueda lineal:
#
#   busqueda_lineal:         Encuentra la PRIMERA ocurrencia.
#   busqueda_lineal_todas:   Encuentra TODAS las ocurrencias.
#   busqueda_lineal_condicion: Busca el primer elemento que
#                              cumple una condición arbitraria.
#
# La variante con condición es muy poderosa porque permite
# buscar por cualquier criterio, no solo igualdad exacta.

def busqueda_lineal(lista: list, objetivo) -> tuple:
    """
    Busca 'objetivo' en 'lista' de forma secuencial.

    Returns:
        (indice, comparaciones) si se encontró;
        (-1, comparaciones) si no se encontró.
    """
    for i, elemento in enumerate(lista):
        if elemento == objetivo:
            return i, i + 1   # Índice encontrado, comparaciones realizadas
    return -1, len(lista)     # No encontrado, se revisó toda la lista


def busqueda_lineal_todas(lista: list, objetivo) -> list:
    """
    Retorna una lista con TODOS los índices donde aparece 'objetivo'.
    Útil cuando puede haber duplicados.
    """
    return [i for i, x in enumerate(lista) if x == objetivo]


def busqueda_lineal_condicion(lista: list, condicion) -> tuple:
    """
    Busca el primer elemento que cumple la condición dada.

    Args:
        lista:     Lista de elementos.
        condicion: Función que recibe un elemento y retorna bool.
    Returns:
        (elemento, indice) si se encontró; (None, -1) si no.
    """
    for i, elemento in enumerate(lista):
        if condicion(elemento):
            return elemento, i
    return None, -1


# --- Demo ---
datos_b = [45, 12, 78, 34, 12, 90, 12, 56, 23]

print("=" * 60)
print(" B1 — Búsqueda Lineal")
print("=" * 60)
print(f"  Lista: {datos_b}")
print()

# Primera ocurrencia de 12
idx, comp = busqueda_lineal(datos_b, 12)
print(f"  busqueda_lineal(12):")
print(f"    Encontrado en índice: {idx}  |  Comparaciones: {comp}")

# Todas las ocurrencias de 12
todos = busqueda_lineal_todas(datos_b, 12)
print(f"\n  busqueda_lineal_todas(12):")
print(f"    Índices encontrados: {todos}")

# Buscar el primer número mayor que 70
elem, idx2 = busqueda_lineal_condicion(datos_b, lambda x: x > 70)
print(f"\n  busqueda_lineal_condicion(x > 70):")
print(f"    Primer elemento > 70: {elem}  en índice {idx2}")

# Elemento ausente
idx3, comp3 = busqueda_lineal(datos_b, 999)
print(f"\n  busqueda_lineal(999) — no existe:")
print(f"    Resultado: {idx3}  |  Se revisaron los {comp3} elementos")

 B1 — Búsqueda Lineal
  Lista: [45, 12, 78, 34, 12, 90, 12, 56, 23]

  busqueda_lineal(12):
    Encontrado en índice: 1  |  Comparaciones: 2

  busqueda_lineal_todas(12):
    Índices encontrados: [1, 4, 6]

  busqueda_lineal_condicion(x > 70):
    Primer elemento > 70: 78  en índice 2

  busqueda_lineal(999) — no existe:
    Resultado: -1  |  Se revisaron los 9 elementos


In [ ]:
# ============================================================
# B1 — Paso 2: Análisis de posición del elemento
# ============================================================
# El número de comparaciones en búsqueda lineal depende de DÓNDE
# está el elemento:
#   - Inicio de la lista  → 1 comparación (mejor caso)
#   - Final de la lista   → n comparaciones (peor caso)
#   - Posición aleatoria  → n/2 comparaciones (promedio)

N = 10000
lista_grande = list(range(N))  # [0, 1, 2, ..., 9999]

casos = [
    ("Inicio",   0,         "Mejor caso   O(1)"),
    ("25%",      N // 4,    "Primer cuartil"),
    ("Centro",   N // 2,    "Caso promedio O(n/2)"),
    ("75%",      3*N // 4,  "Tercer cuartil"),
    ("Final",    N - 1,     "Peor caso     O(n)"),
    ("Ausente",  N + 1,     "No existe     O(n)"),
]

print("=" * 65)
print(f" B1 — Comparaciones según posición del elemento  (n={N})")
print("=" * 65)
print(f"  {'Caso':<10}  {'Objetivo':>8}  {'Comparaciones':>14}  {'% de n':>8}  Descripción")
print("-" * 65)

for nombre, objetivo, descripcion in casos:
    _, comp = busqueda_lineal(lista_grande, objetivo)
    pct = comp / N * 100
    print(f"  {nombre:<10}  {objetivo:>8}  {comp:>14}  {pct:>7.1f}%  {descripcion}")

print()
print("📌 La búsqueda lineal es O(n) en promedio.")
print("   Es la única opción cuando la lista NO está ordenada.")

 B1 — Comparaciones según posición del elemento  (n=10000)
  Caso        Objetivo   Comparaciones    % de n  Descripción
-----------------------------------------------------------------
  Inicio             0               1      0.0%  Mejor caso   O(1)
  25%             2500            2501     25.0%  Primer cuartil
  Centro          5000            5001     50.0%  Caso promedio O(n/2)
  75%             7500            7501     75.0%  Tercer cuartil
  Final           9999           10000    100.0%  Peor caso     O(n)
  Ausente        10001           10000    100.0%  No existe     O(n)

📌 La búsqueda lineal es O(n) en promedio.
   Es la única opción cuando la lista NO está ordenada.


## 🖊️ Documentación Estudiantil — B1 Búsqueda Lineal

> Edita esta celda haciendo doble clic.

---

### 1. ¿Cuándo es la búsqueda lineal la mejor opción?

```
Se considera la mejor opción cuando los datos están desordenados y la búsqueda se realizará muy pocas veces. En ese escenario, revisar la lista elemento por elemento O(n) toma mucho menos tiempo y esfuerzo que ordenar toda la colección primero O(n log n) solo para aplicar un método más avanzado.
```

### 2. Variante con condición
*Escribe un ejemplo de uso de `busqueda_lineal_condicion` con una condición diferente:*

```python
# Ejemplo: Búsqueda del Primer Valor en un Rango
# busqueda_lineal_condicion(lista, lambda x: x >= 100 and x <= 500)
# Encuentra: El primer número que se encuentre dentro del rango de 100 a 500.
```

### 3. Análisis de comparaciones
*¿Qué relación observas entre la posición del elemento y las comparaciones necesarias?*

```
Existe una relación directamente proporcional, ya que el número de comparaciones depende estrictamente de la ubicación física del elemento. Si el objetivo está al inicio, se requiere una sola comparación; si está al final o no existe, el algoritmo se ve obligado a revisar la lista completa.
```

---
# 🟢 B2 — Búsqueda Binaria

## Concepto
La búsqueda binaria **requiere una lista ordenada**. Compara el objetivo con el elemento
**central** y descarta la mitad donde el objetivo no puede estar.

```
Buscar 23 en [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
              ↑_________________________↑
  Centro = 16, 23 > 16 → buscar en mitad derecha
                         [23, 38, 56, 72, 91]
  Centro = 56, 23 < 56 → buscar en mitad izquierda
                         [23, 38]
  Centro = 23 = 23  → ¡Encontrado! (3 comparaciones vs 6 lineal)
```

Con n=1,000,000 elementos, la búsqueda binaria necesita máximo **20 comparaciones**.

In [ ]:
# ============================================================
# B2 — Paso 1: Búsqueda Binaria — versión iterativa y recursiva
# ============================================================
# Implementamos DOS versiones:
#
#   busqueda_binaria_iter:  Iterativa — usa un bucle while.
#                           Preferida en producción (sin overhead de pila).
#   busqueda_binaria_recur: Recursiva — más cercana a la definición
#                           matemática, útil para estudiar la lógica.
#
# INVARIANTE DEL BUCLE:
#   Si el objetivo existe, siempre está en lista[bajo..alto].
#   En cada iteración, el rango se reduce a la mitad.
#   Cuando bajo > alto, el elemento no existe.

def busqueda_binaria_iter(lista: list, objetivo) -> tuple:
    """
    Búsqueda binaria iterativa en una lista ORDENADA.

    Args:
        lista:   Lista ordenada de menor a mayor.
        objetivo: Valor a buscar.
    Returns:
        (indice, comparaciones) si encontrado;
        (-1, comparaciones) si no.
    """
    bajo  = 0
    alto  = len(lista) - 1
    comp  = 0

    while bajo <= alto:
        # Evitar overflow: usar bajo + (alto-bajo)//2 en vez de (bajo+alto)//2
        medio = bajo + (alto - bajo) // 2
        comp += 1

        if lista[medio] == objetivo:
            return medio, comp          # ¡Encontrado!
        elif lista[medio] < objetivo:
            bajo = medio + 1            # Buscar en mitad derecha
        else:
            alto = medio - 1            # Buscar en mitad izquierda

    return -1, comp                     # No encontrado


def busqueda_binaria_recur(lista: list, objetivo, bajo: int = 0,
                           alto: int = None, comp: int = 0) -> tuple:
    """
    Búsqueda binaria recursiva en una lista ORDENADA.
    Cada llamada recursiva reduce el problema a la mitad.
    """
    if alto is None:
        alto = len(lista) - 1

    if bajo > alto:
        return -1, comp   # Caso base: rango vacío → no encontrado

    medio = bajo + (alto - bajo) // 2
    comp += 1

    if lista[medio] == objetivo:
        return medio, comp
    elif lista[medio] < objetivo:
        return busqueda_binaria_recur(lista, objetivo, medio + 1, alto, comp)
    else:
        return busqueda_binaria_recur(lista, objetivo, bajo, medio - 1, comp)


# Demo
lista_ord = sorted(generar_lista(20))

print("=" * 60)
print(" B2 — Búsqueda Binaria")
print("=" * 60)
print(f"  Lista ordenada: {lista_ord}")
print()

objetivo = lista_ord[7]   # Elemento existente
idx_i, comp_i = busqueda_binaria_iter(lista_ord, objetivo)
idx_r, comp_r = busqueda_binaria_recur(lista_ord, objetivo)
print(f"  Buscar {objetivo} (existe):")
print(f"    Iterativa:  índice={idx_i}  comparaciones={comp_i}")
print(f"    Recursiva:  índice={idx_r}  comparaciones={comp_r}")

objetivo_no = 999
idx_n, comp_n = busqueda_binaria_iter(lista_ord, objetivo_no)
print(f"\n  Buscar {objetivo_no} (no existe):")
print(f"    Iterativa:  índice={idx_n}  comparaciones={comp_n}")

 B2 — Búsqueda Binaria
  Lista ordenada: [4, 4, 5, 12, 12, 14, 15, 18, 28, 29, 32, 36, 55, 70, 76, 82, 87, 95, 95, 95]

  Buscar 18 (existe):
    Iterativa:  índice=7  comparaciones=4
    Recursiva:  índice=7  comparaciones=4

  Buscar 999 (no existe):
    Iterativa:  índice=-1  comparaciones=5


In [ ]:
# ============================================================
# B2 — Paso 2: Traza del proceso de reducción de rango
# ============================================================
# Visualizamos cómo el rango [bajo, alto] se reduce a la mitad
# en cada paso, hasta encontrar el elemento o agotar el rango.

def busqueda_binaria_traza(lista: list, objetivo) -> None:
    """Ejecuta búsqueda binaria imprimiendo cada paso del proceso."""
    bajo  = 0
    alto  = len(lista) - 1
    paso  = 1

    print(f"  Buscando {objetivo} en lista de {len(lista)} elementos")
    print(f"  {'Paso':>4}  {'bajo':>5}  {'alto':>5}  {'medio':>6}  {'lista[medio]':>13}  Decisión")
    print("  " + "-" * 58)

    while bajo <= alto:
        medio = bajo + (alto - bajo) // 2
        val   = lista[medio]

        if val == objetivo:
            decision = f"¡Encontrado en índice {medio}!"
            print(f"  {paso:>4}  {bajo:>5}  {alto:>5}  {medio:>6}  {val:>13}  {decision}")
            return
        elif val < objetivo:
            decision = f"lista[{medio}]={val} < {objetivo} → buscar derecha"
            bajo = medio + 1
        else:
            decision = f"lista[{medio}]={val} > {objetivo} → buscar izquierda"
            alto = medio - 1

        print(f"  {paso:>4}  {bajo:>5}  {alto:>5}  {medio:>6}  {val:>13}  {decision}")
        paso += 1

    print(f"  → bajo ({bajo}) > alto ({alto}): elemento NO encontrado")


lista_traza = sorted(generar_lista(30))
print("=" * 65)
print(" B2 — Búsqueda Binaria: traza paso a paso")
print("=" * 65)
print()
busqueda_binaria_traza(lista_traza, lista_traza[21])  # Elemento existente
print()
busqueda_binaria_traza(lista_traza, 9999)              # Elemento ausente

 B2 — Búsqueda Binaria: traza paso a paso

  Buscando 78 en lista de 30 elementos
  Paso   bajo   alto   medio   lista[medio]  Decisión
  ----------------------------------------------------------
     1     15     29      14             36  lista[14]=36 < 78 → buscar derecha
     2     15     21      22             82  lista[22]=82 > 78 → buscar izquierda
     3     19     21      18             70  lista[18]=70 < 78 → buscar derecha
     4     21     21      20             76  lista[20]=76 < 78 → buscar derecha
     5     21     21      21             78  ¡Encontrado en índice 21!

  Buscando 9999 en lista de 30 elementos
  Paso   bajo   alto   medio   lista[medio]  Decisión
  ----------------------------------------------------------
     1     15     29      14             36  lista[14]=36 < 9999 → buscar derecha
     2     23     29      22             82  lista[22]=82 < 9999 → buscar derecha
     3     27     29      26             92  lista[26]=92 < 9999 → buscar derecha
     4 

In [ ]:
# ============================================================
# B2 — Paso 3: Lineal vs Binaria — comparación práctica
# ============================================================
# Comparamos el NÚMERO DE COMPARACIONES de búsqueda lineal
# vs binaria para distintos tamaños de lista.
# Esto ilustra la diferencia entre O(n) y O(log n).

import math

tamanios_b = [10, 100, 1_000, 10_000, 100_000, 1_000_000]

print("=" * 70)
print(" B1 vs B2 — Comparaciones teóricas (peor caso)")
print("=" * 70)
print(f"  {'n':>10}  {'Lineal O(n)':>14}  {'Binaria O(log n)':>18}  {'Reducción':>12}")
print("-" * 70)

for n in tamanios_b:
    lineal  = n
    binaria = math.ceil(math.log2(n + 1))
    factor  = lineal / binaria
    print(f"  {n:>10,}  {lineal:>14,}  {binaria:>18}  {factor:>10,.0f}x")

print()
print("📌 Con 1,000,000 de elementos:")
print("   Lineal necesita hasta 1,000,000 comparaciones.")
print("   Binaria necesita máximo 20 comparaciones.")
print("   La condición: la lista DEBE estar ordenada.")

 B1 vs B2 — Comparaciones teóricas (peor caso)
           n     Lineal O(n)    Binaria O(log n)     Reducción
----------------------------------------------------------------------
          10              10                   4           2x
         100             100                   7          14x
       1,000           1,000                  10         100x
      10,000          10,000                  14         714x
     100,000         100,000                  17       5,882x
   1,000,000       1,000,000                  20      50,000x

📌 Con 1,000,000 de elementos:
   Lineal necesita hasta 1,000,000 comparaciones.
   Binaria necesita máximo 20 comparaciones.
   La condición: la lista DEBE estar ordenada.


## 🖊️ Documentación Estudiantil — B2 Búsqueda Binaria

> Edita esta celda haciendo doble clic.

---

### 1. ¿Por qué la lista debe estar ordenada?
*¿Qué ocurriría si aplicaras búsqueda binaria en una lista desordenada?*

```
[ESCRIBE AQUÍ]
```

### 2. ¿Cómo se calcula el número máximo de comparaciones?
*Para una lista de n=1024 elementos, ¿cuántas comparaciones necesita en el peor caso?*

```
[ESCRIBE AQUÍ — Pista: ¿cuántas veces puedes dividir 1024 a la mitad hasta llegar a 1?]
```

### 3. Diferencias entre versión iterativa y recursiva

| | Iterativa | Recursiva |
|-|-----------|----------|
| Legibilidad | | |
| Uso de pila | | |
| Preferida en producción | | |

### 4. ¿Cuándo es mejor pagar el costo de ordenar primero?
*Si tienes una lista desordenada y necesitas buscar muchas veces, ¿conviene ordenarla?*

```
[ESCRIBE AQUÍ]
```

---
# 🔵 B3 — Búsqueda por Tabla Hash

## Concepto
Una tabla hash usa una **función hash** para convertir la clave en un índice de arreglo.
Si no hay colisiones, la búsqueda, inserción y eliminación son todas **O(1)**.

```
hash("python") → índice 3  →  tabla[3] = "python"
hash("datos")  → índice 7  →  tabla[7] = "datos"

Buscar "python":  hash("python") → 3  →  tabla[3] ¿= "python"? ✓
```

**Colisión:** dos claves producen el mismo índice. Se resuelve con:
- **Encadenamiento:** cada celda tiene una lista enlazada de pares (clave, valor).
- **Direccionamiento abierto:** buscar la siguiente celda libre.

In [ ]:
# ============================================================
# B3 — Paso 1: Tabla Hash con encadenamiento (desde cero)
# ============================================================
# Implementamos una tabla hash completa SIN usar dict de Python.
# Esto permite entender el mecanismo interno.
#
# ESTRUCTURA:
#   - Un arreglo de 'capacidad' celdas (cubetas).
#   - Cada celda es una lista de pares (clave, valor) → encadenamiento.
#   - hash_index(clave) = hash(clave) % capacidad
#
# FACTOR DE CARGA: len(tabla) / capacidad
#   - Bajo (< 0.7): pocas colisiones, búsqueda O(1).
#   - Alto (> 0.9): muchas colisiones, búsqueda degrada a O(n).
#   - Cuando supera 0.75, hacemos REHASH: duplicar capacidad
#     y reinsertar todos los elementos.

class TablaHash:
    """
    Tabla hash con resolución de colisiones por encadenamiento.
    Soporta claves de cualquier tipo hasheable (str, int, tuple).
    """
    FACTOR_CARGA_MAX = 0.75  # Umbral para rehashing

    def __init__(self, capacidad: int = 11):
        # Iniciamos con capacidad primo (reduce colisiones)
        self.capacidad = capacidad
        self.cubetas   = [[] for _ in range(capacidad)]  # Lista de listas
        self._tamanio  = 0
        self.colisiones_totales = 0

    def _hash_index(self, clave) -> int:
        """Convierte una clave en un índice de cubeta."""
        return hash(clave) % self.capacidad

    def insertar(self, clave, valor) -> None:
        """Inserta o actualiza el par (clave, valor). Complejidad: O(1) amortizado."""
        if self.factor_carga() >= self.FACTOR_CARGA_MAX:
            self._rehash()  # Ampliar la tabla antes de insertar

        idx     = self._hash_index(clave)
        cubeta  = self.cubetas[idx]

        # Buscar si la clave ya existe → actualizar
        for i, (k, v) in enumerate(cubeta):
            if k == clave:
                cubeta[i] = (clave, valor)  # Actualizar valor existente
                return

        # Clave nueva: agregar al encadenamiento
        if cubeta:   # La cubeta ya tiene algo → colisión
            self.colisiones_totales += 1
        cubeta.append((clave, valor))
        self._tamanio += 1

    def buscar(self, clave) -> tuple:
        """Busca una clave. Retorna (valor, comparaciones) o (None, comp)."""
        idx    = self._hash_index(clave)
        cubeta = self.cubetas[idx]
        comp   = 0

        for k, v in cubeta:
            comp += 1
            if k == clave:
                return v, comp   # Encontrado

        return None, comp        # No encontrado

    def eliminar(self, clave) -> bool:
        """Elimina una clave. Retorna True si existía, False si no."""
        idx    = self._hash_index(clave)
        cubeta = self.cubetas[idx]

        for i, (k, v) in enumerate(cubeta):
            if k == clave:
                cubeta.pop(i)
                self._tamanio -= 1
                return True
        return False

    def factor_carga(self) -> float:
        """Factor de carga = elementos / capacidad. Indica qué tan llena está la tabla."""
        return self._tamanio / self.capacidad

    def _rehash(self) -> None:
        """
        Duplica la capacidad y reinsertar todos los elementos.
        Esto mantiene el factor de carga bajo y las operaciones O(1).
        """
        nueva_cap  = self.capacidad * 2 + 1   # +1 para mantener impar
        nuevas_cub = [[] for _ in range(nueva_cap)]

        for cubeta in self.cubetas:
            for clave, valor in cubeta:
                idx = hash(clave) % nueva_cap
                nuevas_cub[idx].append((clave, valor))

        self.capacidad = nueva_cap
        self.cubetas   = nuevas_cub
        print(f"  [Rehash] Capacidad aumentada a {nueva_cap}")

    def __len__(self):  return self._tamanio

    def estadisticas(self):
        """Imprime estadísticas de la tabla hash."""
        usadas  = sum(1 for c in self.cubetas if c)
        max_cad = max((len(c) for c in self.cubetas), default=0)
        print(f"  Capacidad:         {self.capacidad}")
        print(f"  Elementos:         {self._tamanio}")
        print(f"  Factor de carga:   {self.factor_carga():.3f}")
        print(f"  Cubetas usadas:    {usadas} / {self.capacidad}")
        print(f"  Mayor cadena:      {max_cad} elementos")
        print(f"  Colisiones totales:{self.colisiones_totales}")


print("✅ TablaHash definida.")

✅ TablaHash definida.


In [ ]:
# ============================================================
# B3 — Paso 2: Demo completa de la Tabla Hash
# ============================================================
# Insertamos datos, realizamos búsquedas, eliminamos y observamos
# las estadísticas de la tabla en distintos momentos.

# --- Caso 1: Índice de palabras ---
print("=" * 60)
print(" B3 — Tabla Hash: índice de palabras → frecuencias")
print("=" * 60)

texto = "el gato come el ratón el gato duerme el perro ladra y el gato"
palabras = texto.split()

frecuencias = TablaHash(capacidad=11)
for palabra in palabras:
    valor_actual, _ = frecuencias.buscar(palabra)
    frecuencias.insertar(palabra, (valor_actual or 0) + 1)

print(f"  Texto: '{texto}'")
print()
print("  Frecuencias:")
for p in sorted(set(palabras)):
    freq, comp = frecuencias.buscar(p)
    print(f"    '{p}': {freq} veces  (comparaciones en búsqueda: {comp})")

print()
print("  Estadísticas de la tabla:")
frecuencias.estadisticas()

# --- Caso 2: Búsqueda vs no existente ---
print()
print("-" * 60)
val, c = frecuencias.buscar("serpiente")
print(f"  Buscar 'serpiente' (no existe): valor={val}  comparaciones={c}")

# --- Caso 3: Eliminar ---
frecuencias.eliminar("el")
val2, c2 = frecuencias.buscar("el")
print(f"  Después de eliminar 'el': valor={val2}  comparaciones={c2}")

 B3 — Tabla Hash: índice de palabras → frecuencias
  Texto: 'el gato come el ratón el gato duerme el perro ladra y el gato'

  Frecuencias:
    'come': 1 veces  (comparaciones en búsqueda: 1)
    'duerme': 1 veces  (comparaciones en búsqueda: 1)
    'el': 5 veces  (comparaciones en búsqueda: 1)
    'gato': 3 veces  (comparaciones en búsqueda: 1)
    'ladra': 1 veces  (comparaciones en búsqueda: 1)
    'perro': 1 veces  (comparaciones en búsqueda: 2)
    'ratón': 1 veces  (comparaciones en búsqueda: 1)
    'y': 1 veces  (comparaciones en búsqueda: 1)

  Estadísticas de la tabla:
  Capacidad:         11
  Elementos:         8
  Factor de carga:   0.727
  Cubetas usadas:    7 / 11
  Mayor cadena:      2 elementos
  Colisiones totales:1

------------------------------------------------------------
  Buscar 'serpiente' (no existe): valor=None  comparaciones=0
  Después de eliminar 'el': valor=None  comparaciones=1


In [ ]:
# ============================================================
# B3 — Paso 3: Comparación Lineal vs Binaria vs Hash
# ============================================================
# Comparamos los tres métodos de búsqueda en términos de
# tiempo real y número de comparaciones.

N = 50_000
datos_busq = generar_lista(N, rango=(1, N*2))
datos_ord  = sorted(datos_busq)

# Construir tabla hash con los mismos datos
tabla = TablaHash(capacidad=N // 3)
for x in datos_busq:
    tabla.insertar(x, True)

# Elegir 10 objetivos: 5 que existen, 5 que no
objetivos_si = random.sample(datos_busq, 5)
objetivos_no = [x for x in range(N*2+1, N*2+6)]
todos_obj    = [(o, True) for o in objetivos_si] + [(o, False) for o in objetivos_no]

print("=" * 75)
print(f" Comparación Búsqueda Lineal vs Binaria vs Hash  (n={N:,})")
print("=" * 75)
print(f"  {'Objetivo':>8}  {'Existe':>6}  {'Lineal':>8}  {'Binaria':>8}  {'Hash':>6}")
print("-" * 75)

total_l = total_b = total_h = 0
for obj, existe in todos_obj:
    _, cl = busqueda_lineal(datos_busq, obj)
    _, cb = busqueda_binaria_iter(datos_ord, obj)
    _, ch = tabla.buscar(obj)
    total_l += cl; total_b += cb; total_h += ch
    ex = "✓" if existe else "✗"
    print(f"  {obj:>8}   {ex:>5}  {cl:>8,}  {cb:>8}  {ch:>6}")

print("-" * 75)
print(f"  {'TOTAL':>8}         {total_l:>8,}  {total_b:>8}  {total_h:>6}")
print()
print("  Estadísticas de la tabla hash:")
tabla.estadisticas()
print()
print("📌 Hash hace 1-2 comparaciones independientemente del tamaño de la colección.")

  [Rehash] Capacidad aumentada a 33333
  [Rehash] Capacidad aumentada a 66667
 Comparación Búsqueda Lineal vs Binaria vs Hash  (n=50,000)
  Objetivo  Existe    Lineal   Binaria    Hash
---------------------------------------------------------------------------
     16261       ✓    12,021        14       1
     51382       ✓    14,223        16       1
     11153       ✓    49,284        15       1
     97056       ✓    15,024        13       1
     32115       ✓    41,862        15       1
    100001       ✗    50,000        16       0
    100002       ✗    50,000        16       0
    100003       ✗    50,000        16       0
    100004       ✗    50,000        16       0
    100005       ✗    50,000        16       0
---------------------------------------------------------------------------
     TOTAL          382,414       153       5

  Estadísticas de la tabla hash:
  Capacidad:         66667
  Elementos:         39359
  Factor de carga:   0.590
  Cubetas usadas:    34153 / 666

## 🖊️ Documentación Estudiantil — B3 Búsqueda Hash

> Edita esta celda haciendo doble clic.

---

### 1. ¿Qué es una colisión y cómo la maneja el encadenamiento?

```
[ESCRIBE AQUÍ]
```

### 2. ¿Qué es el factor de carga y por qué importa?
*¿Qué ocurre cuando el factor de carga supera 0.75?*

```
[ESCRIBE AQUÍ]
```

### 3. Tabla comparativa final de los tres métodos de búsqueda
*Completa con los datos del benchmark y añade tu análisis:*

| | Lineal | Binaria | Hash |
|-|--------|---------|------|
| Comparaciones (n=50,000) | 382.414 | 153 | 5 |
| Requiere ordenamiento | No necesita orden previo en los datos | Es obligatorio que los datos esten ordenados | No necesita orden de menor a mayor en los datos |
| Memoria extra | No necesita almacenamiento extra | No necesita almacenamiento extra | Requiere almacenamiento extra para las casillas de la tabla |
| ¿Cuándo usarla? | En conjunto de datos pequeños o en una cantidad de busqueda mínima | Cuando los datos estan ordenados y la memoria disponible | Cuando se necesita busquedas másivas y constante con maxima velocidad. |

### 4. ¿Cuándo NO usar una tabla hash?
*Piensa en los casos donde las otras opciones son mejores:*

```
[ La Tabla hash no es recomendable cuando el sistema rige bajo restricciones estrictas de memoria, ya que se necesita memoria extra para resevar casillas vacías. Por ejemplo, la busqueda binaria es mejor para situaciones como almacenamiento limitado, identificar minimos, maximos o secuencias ordenadas.

La Tabla Hash no ayuda en operaciones de un solo uso, donde el tiempo de uso de la computadora supera  ]
```

---
# 📊 Resumen Final

## Ordenamiento

| Algoritmo | Mejor | Promedio | Peor | Espacio | Estable | In-place |
|-----------|-------|----------|------|---------|---------|----------|
| Bubble Sort | O(n) | O(n²) | O(n²) | O(1) | ✅ | ✅ |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) | ✅ | ❌ |
| Quick Sort | O(n log n) | O(n log n) | O(n²) | O(log n) | ❌ | ✅ |

## Búsqueda

| Algoritmo | Mejor | Promedio | Peor | Requiere | Espacio |
|-----------|-------|----------|------|----------|---------|
| Lineal | O(1) | O(n) | O(n) | Nada | O(1) |
| Binaria | O(1) | O(log n) | O(log n) | Ordenada | O(1) |
| Hash | O(1) | O(1) | O(n) | Hash func | O(n) |

---

## 🖊️ Reflexión Final del Estudiante

> Completa estas preguntas de integración.

**1. Si tienes una lista de 10 millones de registros que se ordenan una vez y se consultan millones de veces, ¿qué combinación de algoritmos usarías?**
```
[ESCRIBE AQUÍ]
```

**2. ¿En qué situaciones preferiría Bubble Sort sobre Merge Sort?**
```
[ESCRIBE AQUÍ]
```

**3. ¿Qué garantías ofrece Merge Sort que Quick Sort no ofrece?**
```
[ESCRIBE AQUÍ]
```

---
*Guía desarrollada para la materia Estructura de Datos — Python 3.10+*